In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import sys
from tqdm import tqdm
import plotnine as gg

sys.path.append("/workspace")


from src.evaluation.kernel_evaluation import process_and_align, wide_to_long

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)

In [ ]:
dmat_file = "/workspace/results/ecoli_rich_medium/pred/fba_gene_graph_beta_auto/distances.pkl"
metabolic_distances = pd.read_pickle(dmat_file)

dmat_file = "/workspace/results/ecoli_rich_medium/targets/mmd_distances.pkl"
ref_distances = pd.read_pickle(dmat_file)

pred_dist_sub, target_dist_sub = process_and_align(metabolic_distances, ref_distances)

# metric: percentile of distances in target for smallest elements in pred
pred_dist_sub_long = wide_to_long(
    pred_dist_sub,
    "distance_pred",
    "gene1",
    "gene2",
    remove_diagonal=True,
    remove_lower_triangle=True,
)
target_dist_sub_long = wide_to_long(
    target_dist_sub,
    "distance_target",
    "gene1",
    "gene2",
    remove_diagonal=True,
    remove_lower_triangle=True,
)
joint_long = pd.merge(
    pred_dist_sub_long, target_dist_sub_long, on=["gene1", "gene2"], how="inner"
).assign(pair_name=lambda x: x["gene1"] + "_" + x["gene2"])

In [ ]:
query = joint_long.sort_values("distance_pred").loc[lambda x: x["distance_target"] > 0.1].head(500)
query

In [ ]:
lfcs = []
pairs = []

for _, pair_info in tqdm(query.iterrows()):
    gene1 = pair_info["gene1"]
    gene2 = pair_info["gene2"]
    pair_name = f"{gene1}_{gene2}"
    ad_sub = adata[adata.obs["gene"].isin([gene1, gene2])].copy()
    try:
        sc.tl.rank_genes_groups(
            ad_sub,
            groupby="gene",
            groups=[gene1],
            reference=gene2,
            method="wilcoxon",
            use_raw=False,
        )
    except:
        continue
    rg = ad_sub.uns["rank_genes_groups"]
    genes = pd.Index(rg["names"][gene1])
    lfc = pd.Series(rg["logfoldchanges"][gene1], index=genes, name=pair_name)

    lfcs.append(lfc)
    pairs.append(pair_name)
lfcs = np.array(lfcs)

In [ ]:
pairs

In [ ]:
abs_lfcs = np.abs(lfcs)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

tsne = TSNE(n_components=2, random_state=42)
abs_lfcs_ = PCA(n_components=50).fit_transform(abs_lfcs)
adata_tsne = tsne.fit_transform(abs_lfcs_)
adata_tsne = pd.DataFrame(adata_tsne, index=pairs)
adata_tsne.columns = ["tsne1", "tsne2"]
adata_tsne = adata_tsne.merge(joint_long, left_index=True, right_on="pair_name")

In [ ]:
(
    gg.ggplot(adata_tsne, gg.aes(x="tsne1", y="tsne2", color="distance_target"))
    + gg.geom_point()
    + gg.theme_minimal()
    # + gg.scale_color_continuous(limits=(0, 0.1))
)

In [ ]:
import plotly.express as px

fig = px.scatter(
    adata_tsne,
    x="tsne1",
    y="tsne2",
    color="distance_target",
    hover_data=["pair_name"],
)
fig.update_layout(template="plotly_white", width=750, height=750)
# fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()